> 📅 __Date: 2026-08-28__

# 🔎 **RAG Architecture**

> **Goal:** Understand why LLMs need external context, how Retrieval-Augmented Generation (RAG) solves private-data and context-window problems, how documents are indexed, how relevant chunks are retrieved at runtime, and how retrieved context is supplied to the LLM.

---

# 🗺️ **RAG Learning Roadmap**

```text
LLM
 ↓
Problem Statement
 ↓
LLM Limitations
 ↓
In-Context Learning
 ↓
Document Loading
 ↓
Context Window Problem
 ↓
Chunking
 ↓
Embeddings
 ↓
Vector Database
 ↓
Indexing / Ingestion Pipeline
 ↓
Retriever
 ↓
Runtime Retrieval
 ↓
Augmentation
 ↓
Generation
 ↓
Complete RAG Architecture
```

---

# 🎯 **Problem Statement**

## **Project: Chatbot for Q&A on "Attention Is All You Need" Paper**

**Suppose we want to build a chatbot that can answer questions about:**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

**For example:**

```text
User:
"What is attention?"
       ↓
LLM
       ↓
Answer
```

**A simple model call is:**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import OpenAI

model = OpenAI()

response = model.invoke(
    "What is attention?"
)

print(response)



Attention refers to the ability to focus one's thoughts and senses on a specific task or stimulus, while filtering out distractions. It involves directing mental energy towards a particular goal or object, and being aware of and responsive to relevant information. Attention is a cognitive process that is essential for learning, problem-solving, decision-making, and overall functioning in daily life. It can be voluntary or involuntary, and can vary in intensity and duration. Attention is a fundamental aspect of human consciousness and plays a crucial role in our perception and understanding of the world. 


This works for a normal model query, but the model has **not automatically been given the contents of our PDF**.

That leads to the first problem.

---

# 1️⃣ **LLM is Not Aware of Private / External Data**

> **An LLM is not automatically aware of the private data or documents available inside our application.**

**For example:**

In [3]:
model.invoke(
    "What is the admission process for our internal GenAI program?"
)

"\n\nThe admission process for our internal GenAI program is as follows:\n\n1. Application: Interested employees can apply for the GenAI program by filling out an application form. The form will ask for basic personal information, as well as details about their educational background, relevant work experience, and interest in artificial intelligence.\n\n2. Assessment: Once the application period is closed, the HR team will review all the applications and shortlist candidates based on their qualifications and interest in AI. Shortlisted candidates will be invited to an assessment, which may include a written test, coding challenge, and/or interview.\n\n3. Selection: Based on the assessment results, a final list of candidates will be selected for the program. The number of candidates selected may vary depending on the availability of resources and the program's capacity.\n\n4. Orientation: Selected candidates will attend an orientation session to familiarize themselves with the program s

If the information exists only in private documents, the model cannot reliably answer from those documents unless the application provides that information.

**Conceptually:**

```text
Private Data
     │
     │ ❌ Not automatically available
     ↓
    LLM
     ↓
Potentially Unknown / Incorrect Answer
```

**The application therefore needs a way to provide external context:**

```text
Prompt Context
+
Retrieval
+
RAG
+
Database / Tool Access
```

> **An LLM does not automatically know your application's private data.**

---

# 🧠 **In-Context Learning**

> **In-Context Learning = Providing relevant context along with the question so that the model can use that information while generating the answer.**

**Instead of:**

```text
Question
   ↓
LLM
   ↓
Answer
```

**we provide:**

```text
Context + Question
       ↓
      LLM
       ↓
     Answer
```

---

# 🧩 **In-Context Learning Example**

**Suppose the paper contains information such as:**

```text
Ashish, with Illia, designed and implemented the first Transformer models
and has been crucially involved in every aspect of this work.

Noam proposed scaled dot-product attention, multi-head attention
and the parameter-free position representation and became the other
person involved in nearly every detail.
```

**We can manually provide relevant context:**

In [4]:
prompt = """
Based on the given context, answer the question.

Context:
Ashish, with Illia, designed and implemented the first Transformer models
and has been crucially involved in every aspect of this work. Noam proposed
scaled dot-product attention, multi-head attention and the parameter-free
position representation and became the other person involved in nearly every
detail.

Question:
What is attention?
"""

**Then:**

In [5]:
response = model.invoke(prompt)

print(response)


Attention is a mechanism in neural networks that allows the model to focus on certain parts of the input data, enabling it to better process and understand the information.


**The key idea is:**

```text
Relevant Context
      +
Question
      ↓
     LLM
      ↓
   Answer
```

This basic idea is the foundation of RAG.

---

# 🆚 **Without Context vs With Context**

### **Without Context**

```text
Question
"What is attention?"
       ↓
LLM
       ↓
Answer from model knowledge
```

### **With Context**

```text
Relevant Context
       +
Question
       ↓
LLM
       ↓
Answer using provided context
```

---

# ⚠️ **Problem #2: Our Data is in PDF Format**

**Our knowledge source is:**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

Before the LLM can use this PDF, we first need to extract its contents.

This is where **Document Loaders** are used.

---

# 📚 **Document Loader**

> **Document Loader = A component that loads data from a source and converts it into LangChain `Document` objects.**

**For PDF:**

```text
PDF
 ↓
PyPDFLoader
 ↓
Document Objects
```

**Examples:**

```text
PDF
 → PDF Loader

Web Page
 → Web Loader

CSV
 → CSV Loader

Text File
 → Text Loader
```

---

# 📦 **Install the Required PDF Package**

**For this example:**

```python
%pip install -U pypdf
```

**The LangChain community loader package is also required:**

```python
%pip install -U langchain-community
```

**The general idea is:**

```text
langchain-community
        +
loader-specific library
        ↓
Document Loader
```

---

# 📄 **Load the PDF**

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "assets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

C:\Users\KP\AppData\Local\Temp\ipykernel_17856\1550334054.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load the documents:**


In [7]:
docs = loader.load()

`docs` is a collection of LangChain `Document` objects.

---

# 🧩 **Document Object**

> **Document = A standardized object containing extracted content and associated metadata.**

**A useful mental model is:**

```text
Document
 ├── page_content
 └── metadata
```

### **`page_content`**

> **`page_content` = The extracted textual content.**

```python
docs[0].page_content
```

### **`metadata`**

> **`metadata` = Information associated with the document content or its source.**

**For example:**

In [8]:
docs[0].metadata

{'producer': 'PyPDF2',
 'creator': 'PyPDF',
 'creationdate': '',
 'subject': 'Neural Information Processing Systems http://nips.cc/',
 'publisher': 'Curran Associates, Inc.',
 'language': 'en-US',
 'created': '2017',
 'eventtype': 'Poster',
 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-

may contain source/page information.

---

# 🔍 **Inspect the Loaded Document**

In [9]:
type(docs)

list

In [10]:
type(docs[0])

langchain_core.documents.base.Document

```python
docs[0].page_content
```

```python
docs[0].metadata
```

**Conceptually:**

```text
PDF
 ↓
PyPDFLoader
 ↓
List[Document]
 ↓
 ┌─────────────────────┐
 │ Document            │
 │                     │
 │ page_content        │
 │ metadata            │
 └─────────────────────┘
```

---

# 📝 **Extracting the Complete Text**

**A simple approach is:**

In [11]:
extracted_text = ""

for page in docs:
    extracted_text += page.page_content

# print(extracted_text)

**Now we have:**

```text
PDF
 ↓
Extracted Text
```

We may then try to provide the complete text to the model.

---

# ⚠️ **Problem #3: Context Window Limit**

**Suppose we create:**

In [12]:
prompt = f"""
Based on the given context, answer the question.

Context:
{extracted_text}

Question:
What is attention?
"""

**and then:**

```python
response = model.invoke(prompt)
```

For a large PDF, this can fail because the document is larger than the model's available context window.

---

# 🚨 **Example Context Window Error**

**The request may fail with:**

```text
BadRequestError: Error code: 400 - {
    'error': {
        'message': "This model's maximum context length is 4097 tokens,
        however you requested 8527 tokens
        (8271 in your prompt; 256 for the completion).
        Please reduce your prompt; or completion length.",
        'type': 'invalid_request_error',
        'param': None,
        'code': None
    }
}
```

**The important observation is:**

```text
Requested Tokens
      >
Model Context Limit
```

---

# 🪟 **Context Window**

> **Context Window = The maximum amount of token context that a model can process for a request / generation setup.**

**A simplified constraint is:**

$$
T_{\text{input}} + T_{\text{output}}
\leq
T_{\text{context}}
$$

**where:**

```text
T_input
→ Input tokens

T_output
→ Requested output tokens

T_context
→ Model context limit
```

**If:**

$$
T_{\text{input}} + T_{\text{output}}
>
T_{\text{context}}
$$

the request exceeds the available context.

---

# 🧮 **Understanding the Error**

**Suppose:**

```text
Maximum context = 4097 tokens

Prompt = 8271 tokens

Completion = 256 tokens
```

**Then:**

$$
8271 + 256 = 8527
$$

**Therefore:**

$$
8527 > 4097
$$

So the request cannot be processed under that context limit.

---

# ✂️ **Solution: Split the Document into Smaller Chunks**

**Instead of sending the complete document as one huge prompt:**

```text
Entire Document
      ↓
One Huge Prompt
      ↓
❌ Context Window Error
```

we split it into smaller pieces.

**Example:**

```text
Document
   ↓
Chunking
   ↓
 ┌──────────────┬─────────────────┬──────────────┐
 ↓              ↓                 ↓
Chunk 1       Chunk 2           Chunk 3          ...
 ↓              ↓                 ↓
Self-Attention Positional       Add & Norm
                Encoding
```

**Example chunks:**

```text
Chunk 1 → Self-Attention
Chunk 2 → Positional Encoding
Chunk 3 → Add & Norm
Chunk 4 → Feed Forward
Chunk 5 → Masked Multi-Head Attention
...
```

---

# 📦 **Chunking**

> **Chunking = Splitting a large document into smaller pieces so that those pieces can be processed independently or retrieved selectively.**

**Conceptually:**

```text
Large Document
      ↓
Chunking
      ↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
...
Chunk N
```

---

# ❓ **If Models Have 1M Context Windows, Why Do We Still Need Chunking / RAG?**

A large context window reduces one limitation, but it does not make retrieval unnecessary.

**Two important reasons from this architecture are:**

```text
1. Cost
2. Context Rot
```

---

# 💰 **1. Cost**

**If every question sends the entire document to the model:**

```text
Entire Document
      +
User Query
      ↓
LLM
```

many tokens may be processed on every request.

**Conceptually:**

$$
\text{Cost}
\propto
\text{Tokens Processed}
$$

**Therefore:**

```text
Larger Context
      ↓
More Tokens Processed
      ↓
Potentially Higher Cost
```

RAG instead tries to retrieve only the relevant pieces.

---

# 🧠 **2. Context Rot**

> **Context Rot = A decline in model performance / effectiveness as the amount of supplied context grows.**

**The important idea is:**

```text
More Context
     ≠
More Useful Context
```

Even when a model supports a very large context window, sending a large amount of irrelevant information can make the application less efficient and can reduce how effectively the model uses the context.

**Therefore:**

> **RAG is not only about overcoming context limits; it is also about selecting relevant information.**

---

# 🔎 **Problem #4: Finding the Relevant Chunk**

**Suppose our document contains:**

```text
Chunk 1 → Self-Attention
Chunk 2 → Positional Encoding
Chunk 3 → Add & Norm
Chunk 4 → Feed Forward
Chunk 5 → Masked MHA
...
```

**The user asks:**

```text
"What is self-attention?"
```

We do not need every chunk.

**We want:**

```text
User Query
    ↓
Find Similar / Relevant Chunks
    ↓
Chunk 1 → Self-Attention
```

**So we need a way to calculate similarity between:**

```text
User Query
      ↔
Document Chunks
```

---

# 📐 **Cosine Similarity**

A common similarity measure for vectors is cosine similarity.

**For vectors $a$ and $b$:**

$$
\operatorname{cosine\ similarity}(a,b)
=
\frac{a\cdot b}
{\|a\|\|b\|}
$$

**Conceptually:**

```text
Query Vector
      ↕
Similarity
      ↕
Chunk Vector
```

The chunk with the highest similarity can be considered more relevant.

---

# ⚠️ **But Queries and Chunks are Text**

The cosine similarity formula works on vectors.

**Our data is initially:**

```text
User Query
→ Text

Document Chunk
→ Text
```

So we need to convert text into vectors.

This is the job of **Embedding Models**.

---

# 🧠 **Embedding Models**

> **Embedding Model = A model that converts text into numerical vector representations.**

**Conceptually:**

```text
Text
 ↓
Embedding Model
 ↓
Vector
```

**Example:**

```text
"What is self-attention?"
          ↓
Embedding Model
          ↓
[0.12, -0.41, 0.73, ...]
```

**Likewise:**

```text
"Self-attention lets tokens interact..."
          ↓
Embedding Model
          ↓
[0.10, -0.38, 0.69, ...]
```

Now vectors can be compared.

---

# 🔄 **Text → Embeddings**

**For document chunks:**

```text
Document Chunk
      ↓
Embedding Model
      ↓
Chunk Vector
```

**For the user query:**

```text
User Query
      ↓
Embedding Model
      ↓
Query Vector
```

**Then:**

```text
Query Vector
      ↓
Similarity Search
      ↓
Chunk Vectors
```

---

# 🧮 **RAG Similarity Calculation**

**Let:**

```text
q   = Query embedding
d_i = Embedding of document chunk i
```

**Then:**

$$
\operatorname{sim}(q,d_i)
=
\frac{q\cdot d_i}
{\|q\|\|d_i\|}
$$

**The most similar chunk can be written conceptually as:**

$$
d^*
=
\arg\max_{d_i}
\operatorname{sim}(q,d_i)
$$

For multiple chunks, we retrieve the top-k results.

---

# 🗃️ **Vector Database**

> **Vector Database = A database designed to store and search vector / embedding representations.**

**A useful mental model is:**

```text
Vector Database
 ├── Embedding
 ├── Text
 └── Metadata
```

**Example:**

```text
┌──────────────────────────────────────┐
│ Vector Database                      │
│                                      │
│ Vector      Text        Metadata     │
│ [0.1,...]   Chunk 1     Page 1       │
│ [0.4,...]   Chunk 2     Page 2       │
│ [0.7,...]   Chunk 3     Page 3       │
│ ...                                  │
└──────────────────────────────────────┘
```

The vector is used for similarity search.

The text is returned as context.

The metadata helps identify the source.

---

# 🧩 **Why Store Text and Metadata Along with the Vector?**

The LLM needs readable information, not only numbers.

**Therefore a useful stored record can be thought of as:**

```text
Vector
+
Original Text
+
Metadata
```

**Example:**

```json
{
  "text": "Self-attention allows each token to ...",
  "vector": [0.12, -0.41, 0.73],
  "metadata": {
    "source": "attention-paper.pdf",
    "page": 3
  }
}
```

---

# 🏗️ **RAG Architecture**

**RAG can be divided into two major pipelines:**

```text
1. Indexing / Ingestion Pipeline
2. Runtime / Query Pipeline
```

---

# 🏗️ **1. Indexing / Ingestion Pipeline**

> **Indexing / Ingestion = Preparing documents so that they can be efficiently searched later.**

**Overall flow:**

```text
Document
   ↓
Loader
   ↓
Extracted Text
   ↓
Chunking
   ↓
Chunk 1, Chunk 2, Chunk 3, ...
   ↓
Embedding Model
   ↓
Vectors
   ↓
Vector Database
```

---

# 🧭 **Indexing Pipeline — Detailed**

### **Step 1: Document**

```text
NIPS-2017-attention-is-all-you-need-Paper.pdf
```

### **Step 2: Load**

```text
PDF
 ↓
PyPDFLoader
```

### **Step 3: Extract**

```text
PyPDFLoader
 ↓
Document Objects
 ↓
page_content
```

### **Step 4: Chunk**

```text
Large Document
 ↓
Chunking
 ↓
Chunk 1
Chunk 2
Chunk 3
...
```

### **Step 5: Embed**

```text
Each Chunk
 ↓
Embedding Model
 ↓
Vector
```

### **Step 6: Store**

```text
Vectors
+
Text
+
Metadata
 ↓
Vector DB
```

---

# 🧭 **Indexing Pipeline Diagram**

```text
                    DOCUMENT
                       │
                       ↓
                DOCUMENT LOADER
                       │
                       ↓
                EXTRACTED TEXT
                       │
                       ↓
                    CHUNKING
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       Chunk 1      Chunk 2      Chunk 3
          │            │            │
          └────────────┼────────────┘
                       ↓
                 EMBEDDING MODEL
                       ↓
                     VECTORS
                       ↓
                 VECTOR DATABASE
```

---

# 🔎 **2. Runtime / Query Pipeline**

**The runtime pipeline starts when the user asks a question:**

```text
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Similarity Search
    ↓
Relevant Chunks
```

**Then:**

```text
Relevant Chunks
      +
User Query
      ↓
Prompt / Augmentation
      ↓
LLM
      ↓
Response
```

---

# 🧭 **Runtime Pipeline — Detailed**

### **Step 1: User Query**

```text
"What is self-attention?"
```

### **Step 2: Query Embedding**

```text
User Query
    ↓
Embedding Model
    ↓
Query Vector
```

### **Step 3: Similarity Search**

```text
Query Vector
     ↓
Vector DB
     ↓
Compare with Chunk Vectors
```

### **Step 4: Retrieval**

```text
Similarity Scores
      ↓
Top-k
      ↓
Relevant Chunks
```

### **Step 5: Augmentation**

```text
User Query
+
Retrieved Chunks
 ↓
Augmented Prompt
```

### **Step 6: Generation**

```text
Augmented Prompt
       ↓
      LLM
       ↓
    Answer
```

---

# 🔎 **Retriever**

> **Retriever = A component responsible for retrieving relevant information from the indexed knowledge source.**

**Conceptually:**

```text
User Query
    ↓
Retriever
    ↓
Relevant Chunks
```

**Its job is:**

```text
Find useful information
```

**not:**

```text
Generate the final answer
```

---

# 🧠 **Retriever Mental Model**

```text
Question
   ↓
Retriever
   ↓
"Which pieces of stored knowledge are relevant?"
   ↓
Relevant Chunks
```

**Then:**

```text
Relevant Chunks
      +
Question
      ↓
LLM
      ↓
Answer
```

---

# ✨ **Augmentation**

> **Augmentation = Combining retrieved information with the user's query to create the context / prompt supplied to the LLM.**

**Conceptually:**

```text
User Query
     +
Retrieved Chunks
     ↓
Augmented Prompt
     ↓
LLM
```

**Example:**

```text
Context:
Chunk 1: Self-attention allows...

Chunk 2: The attention function maps...

Question:
What is self-attention?
```

This becomes the model's augmented input.

---

# 🤖 **Generation**

> **Generation = Using the LLM to produce the final answer from the augmented prompt.**

```text
Augmented Prompt
       ↓
      LLM
       ↓
Final Answer
```

**Therefore:**

```text
Retrieve
   ↓
Relevant Context
   ↓
Augment
   ↓
Generate
```

---

# 📖 **RAG = Retrieve + Augment + Generate**

### **Retriever**

> Retrieve relevant information.

```text
Question
 ↓
Retriever
 ↓
Relevant Chunks
```

### **Augmented**

> Combine retrieved information with the user query.

```text
Relevant Chunks
+
Question
 ↓
Augmented Prompt
```

### **Generation**

> Generate the answer using the LLM.

```text
Augmented Prompt
 ↓
LLM
 ↓
Answer
```

**So:**

```text
Retriever
     +
Augmentation
     +
Generation
     ↓
RAG
```

---

# 🖼️ **RAG Architecture**

<div align="center">

<img src="assets/rag.png" width="900" alt="RAG Architecture">

<p><em>Figure: RAG architecture showing the indexing pipeline and runtime retrieval / generation flow.</em></p>

</div>

---

# 🧱 **Components of RAG**

The major components can be organized into four stages:

```text
┌───────────────────────────────────────────────┐
│ INDEXING                                      │
├───────────────────────────────────────────────┤
│ Text Extraction                               │
│ Text Splitting / Chunking                     │
│ Embedding Model                               │
│ Vector Database                               │
└───────────────────────────────────────────────┘

                     ↓

┌───────────────────────────────────────────────┐
│ RETRIEVAL                                     │
├───────────────────────────────────────────────┤
│ Retriever                                     │
└───────────────────────────────────────────────┘

                     ↓

┌───────────────────────────────────────────────┐
│ AUGMENTATION                                  │
├───────────────────────────────────────────────┤
│ Prompt Creation                               │
│ Query + Retrieved Context                     │
└───────────────────────────────────────────────┘

                     ↓

┌───────────────────────────────────────────────┐
│ GENERATION                                    │
├───────────────────────────────────────────────┤
│ LLM                                           │
└───────────────────────────────────────────────┘
```

---

# 📦 **Stage 1 — Text Extraction**

> **Text Extraction = Converting the original source into text that can be processed by downstream RAG components.**

**Example:**

```text
PDF
 ↓
PyPDFLoader
 ↓
Extracted Text
```

---

# ✂️ **Stage 2 — Text Splitting / Chunking**

> **Text Splitting = Dividing a large document into smaller chunks.**

Why?

```text
Large Document
      ↓
Too Large / Too Much Irrelevant Context
      ↓
Chunking
      ↓
Smaller Searchable Units
```

**Example:**

```text
Chunk 1 → Self-Attention
Chunk 2 → Positional Encoding
Chunk 3 → Feed Forward
Chunk 4 → Multi-Head Attention
```

---

# 🧠 **Stage 3 — Embedding Model**

> **Embedding Model = Converts text chunks and queries into vectors so that semantic similarity can be measured.**

**For chunks:**

```text
Chunk
 ↓
Embedding Model
 ↓
Vector
```

**For query:**

```text
Query
 ↓
Embedding Model
 ↓
Query Vector
```

---

# 🗃️ **Stage 4 — Vector Database**

> **Vector Database = Stores embeddings and associated information so relevant content can be searched.**

**A stored record can contain:**

```text
Vector
+
Text
+
Metadata
```

---

# 🔎 **Stage 5 — Retriever**

> **Retriever = Fetches the most relevant stored chunks for a user's query.**

**Conceptually:**

```text
User Query
    ↓
Query Embedding
    ↓
Similarity Search
    ↓
Top-k Chunks
```

The number of selected chunks is represented by `k`.

**Conceptually:**

$$
\text{Retrieved Context}
=
\operatorname{TopK}
\left(
\operatorname{Similarity}(q,D)
\right)
$$

**where:**

```text
q
→ Query vector

D
→ Document-vector collection

TopK
→ Highest-scoring results
```

---

# ✨ **Stage 6 — Prompt Creation / Augmentation**

**The retrieved information is combined with the question:**

```text
Retrieved Chunk 1
+
Retrieved Chunk 2
+
User Query
      ↓
Prompt
```

**A conceptual prompt is:**

```text
Based on the provided context, answer the question.

Context:
{retrieved_context}

Question:
{user_question}
```

---

# 🤖 **Stage 7 — LLM**

**The LLM receives the augmented prompt:**

```text
Augmented Prompt
       ↓
      LLM
       ↓
Answer
```

The LLM is responsible for generating the natural-language response.

---

# 🧠 **Complete RAG Formula**

**A simplified conceptual formulation is:**

$$
\text{Answer}
=
\operatorname{LLM}
\left(
\text{Query}
+
\text{Retrieved Context}
\right)
$$

**The retrieved context is produced by:**

$$
\text{Retrieved Context}
=
\operatorname{Retriever}
\left(
\operatorname{Embed}(\text{Query}),
\operatorname{Index}
\right)
$$

**Therefore:**

$$
\text{RAG}
=
\operatorname{Generate}
\left(
\text{Query},
\operatorname{Retrieve}(\text{Query})
\right)
$$


---

# 🎯 **Why Retrieve Instead of Passing Everything?**

**Suppose the knowledge base contains:**

```text
1000 chunks
```

**but the user asks about:**

```text
Self-Attention
```

**Passing all 1000 chunks means:**

```text
1000 Chunks
   ↓
Huge Context
   ↓
LLM
```

**RAG instead tries to do:**

```text
1000 Chunks
   ↓
Similarity Search
   ↓
Top Relevant Chunks
   ↓
LLM
```

**Therefore:**

```text
Knowledge Base
      ↓
Search
      ↓
Relevant Information
      ↓
Generation
```

---

# 🧠 **RAG and Private Data**

One of the most important uses of RAG is working with private / external data.

```text
Private Documents
       ↓
Indexing
       ↓
Vector Database
       ↓
Retriever
       ↓
Relevant Private Context
       ↓
LLM
       ↓
Answer
```

The model does not need that private information to have been part of its original training data.

Instead, the application provides relevant information at runtime.

---

# 📚 **RAG vs In-Context Learning**

These ideas are closely related.

### **In-Context Learning**

```text
Context
  +
Question
  ↓
LLM
```

### **RAG**

```text
Large Knowledge Base
       ↓
Retriever
       ↓
Relevant Context
       +
Question
       ↓
LLM
```

**Therefore:**

> **RAG can be viewed as an automated way of constructing useful context before generation.**

---

# 🆚 **Without RAG vs With RAG**

| Area | Without RAG | With RAG |
|---|---|---|
| **Private Data** | Not automatically available | Retrieved at runtime |
| **Knowledge Source** | Model knowledge / manually supplied context | External indexed knowledge |
| **Large Documents** | Can exceed context limits | Relevant chunks can be selected |
| **Prompt Size** | Can become very large | Focused on retrieved context |
| **Retrieval** | None | Retriever selects relevant chunks |
| **Architecture** | Simple model call | Indexing + retrieval + augmentation + generation |

---

# 🆚 **Traditional LLM Call vs RAG**

### **Traditional LLM Call**

```text
User Question
      ↓
LLM
      ↓
Answer
```

### **RAG**

```text
User Question
      ↓
Embedding
      ↓
Retriever
      ↓
Relevant Chunks
      ↓
Augmented Prompt
      ↓
LLM
      ↓
Answer
```

---

# 🧮 **RAG Cost Mental Model**

**A conceptual RAG request may involve:**

```text
Query Embedding
+
Vector Search
+
Retrieved Context
+
LLM Input Tokens
+
LLM Output Tokens
```

**Therefore:**

$$
\text{RAG Cost}
\approx
\text{Embedding Cost}
+
\text{Retrieval Cost}
+
\text{LLM Input Cost}
+
\text{LLM Output Cost}
$$

The exact cost depends on the model, embedding provider, vector database and application architecture.

---

# 📈 **RAG Quality Mental Model**

**A useful conceptual view is:**

$$
\text{RAG Quality}
\approx
f(
\text{Retrieval Quality},
\text{Context Quality},
\text{Prompt Quality},
\text{Generation Quality}
)
$$

**This means:**

```text
Good Retrieval
+
Relevant Context
+
Good Prompt
+
Good LLM
```

is more likely to produce a useful answer.

---

# 🔬 **Semantic Search Example**

**Suppose we have:**

```text
Query:
"What is self-attention?"
```

**and:**

```text
Chunk 1:
"Self-attention allows every token to attend to other tokens."

Chunk 2:
"Positional encodings provide information about token positions."

Chunk 3:
"The feed-forward network applies transformations independently."
```

**Conceptually:**

```text
Query
 ↓
Similarity Search
 ↓
Chunk 1 → High similarity
Chunk 2 → Lower similarity
Chunk 3 → Lower similarity
 ↓
Chunk 1 selected
```

**Then:**

```text
Relevant Context
      ↓
LLM
      ↓
Answer
```

---

# 🎯 **Top-k Retrieval**

Usually we do not retrieve only one chunk.

**We can retrieve the top `k` results:**

```text
Query
 ↓
Similarity Search
 ↓
Top-k Chunks
```

**Example:**

```text
k = 3

Chunk 7  → 0.91
Chunk 12 → 0.87
Chunk 18 → 0.83
```

**Then:**

```text
Top 3 Chunks
     ↓
Prompt
     ↓
LLM
```

**Conceptually:**

$$
\text{TopK}(q)
=
\{d_{i_1},d_{i_2},\ldots,d_{i_k}\}
$$

---

# 🏷️ **Metadata in RAG**

**Metadata can be useful for:**

```text
Source Filtering
Page Tracking
Document Identification
Debugging
Citations
Access Control
```

**Conceptually:**

```text
Retrieved Chunk
      ↓
Text + Metadata
      ↓
Application
```

---

# 🔄 **RAG Runtime Example**

**Suppose the user asks:**

```text
"What is self-attention?"
```

**The runtime process is:**

```text
User Query
"What is self-attention?"
         ↓
Embedding Model
         ↓
Query Vector
         ↓
Vector DB Search
         ↓
Relevant Chunks
         ↓
Self-Attention Context
         ↓
Augmented Prompt
         ↓
LLM
         ↓
Answer
```

---

# 🧩 **Complete RAG Architecture — Two Pipelines**

**The most important architecture to remember is:**

```text
                    RAG SYSTEM
                       │
        ┌──────────────┴──────────────┐
        ↓                             ↓
     INDEXING                      RUNTIME
        ↓                             ↓
    Document                       Query
        ↓                             ↓
      Loader                    Embedding
        ↓                             ↓
    Chunking                   Query Vector
        ↓                             ↓
   Embedding                   Similarity Search
        ↓                             ↓
    Vector DB                  Relevant Chunks
        │                             ↓
        │                        Augmentation
        │                             ↓
        └────────────────────→ Query + Context
                                      ↓
                                     LLM
                                      ↓
                                   Answer
```

---

# 🧠 **Complete RAG Mental Model**

```text
                         RAG
                          │
          ┌───────────────┴────────────────┐
          │                                │
          ↓                                ↓
      INDEXING                           QUERY
          │                                │
          ↓                                ↓
     DOCUMENTS                         USER QUERY
          ↓                                ↓
       LOADER                         EMBEDDING MODEL
          ↓                                ↓
    EXTRACTED TEXT                    QUERY VECTOR
          ↓                                │
      CHUNKING                             ↓
          ↓                         SIMILARITY SEARCH
       CHUNKS                              ↓
          ↓                         RELEVANT CHUNKS
    EMBEDDING MODEL                        │
          ↓                                │
       VECTORS                             │
          ↓                                │
     VECTOR DATABASE                       │
          └───────────────┬────────────────┘
                          ↓
                    AUGMENTATION
                          ↓
               QUERY + RETRIEVED CONTEXT
                          ↓
                         LLM
                          ↓
                       RESPONSE
```

---

# ⚠️ **Common Beginner Mistakes**

## **1. Thinking RAG Means Training the LLM**

```text
RAG
≠
Fine-Tuning
```

RAG provides external context at runtime.

```text
Knowledge
 ↓
Retriever
 ↓
Context
 ↓
LLM
```

It does not require changing the model's learned weights.

---

## **2. Thinking the Vector DB Stores Only Vectors**

**A useful record can contain:**

```text
Vector
+
Text
+
Metadata
```

---

## **3. Thinking the Retriever Generates the Answer**

```text
Retriever
→ Finds relevant information

LLM
→ Generates the answer
```

---

## **4. Sending the Entire Knowledge Base to the LLM**

```text
Entire Knowledge Base
       ↓
Huge Prompt
       ↓
More Tokens
       ↓
Higher Cost / Context Pressure
```

RAG tries to retrieve only relevant information.

---

## **5. Assuming a Larger Context Window Eliminates RAG**

**A large context window may reduce context-limit problems, but RAG can still help with:**

```text
Cost
Relevance
Context Management
Large Knowledge Bases
Private Data
```

---

## **6. Assuming Better Embeddings Automatically Mean Better RAG**

**A RAG system depends on multiple stages:**

```text
Document Quality
+
Chunking
+
Embeddings
+
Retrieval
+
Prompting
+
LLM
```

A weak stage can reduce overall quality.

---

# 🧠 **Indexing vs Runtime**

### **Indexing**

```text
Documents
 ↓
Loader
 ↓
Chunking
 ↓
Embeddings
 ↓
Vector DB
```

### **Runtime**

```text
User Query
 ↓
Query Embedding
 ↓
Similarity Search
 ↓
Relevant Chunks
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

**Memory trick:**

```text
Indexing
→ Prepare the knowledge

Runtime
→ Retrieve + Generate
```

---

# 🧮 **RAG Similarity Pipeline**

**Let the document chunks be:**

$$
D = \{d_1,d_2,\ldots,d_N\}
$$

**and their embeddings:**

$$
E(D)=\{e_1,e_2,\ldots,e_N\}
$$

**For a user query $q$:**

$$
e_q = E(q)
$$

**Then calculate similarity between:**

$$
e_q
\quad \text{and} \quad
e_i
$$

for each chunk.

The top-scoring chunks are retrieved.

---

# 💡 **Why RAG is Useful**

RAG addresses several practical problems together:

```text
Private / External Data
        ↓
Retrieved at Runtime

Large Documents
        ↓
Chunked + Retrieved Selectively

Text Similarity
        ↓
Embeddings + Similarity Search

Searchable Knowledge
        ↓
Vector Database

Relevant Context
        ↓
Augmented Prompt

Final Response
        ↓
LLM
```

---

# 🎓 **Interview-Friendly Explanation**

> **If an interviewer asks: "What is RAG and how does it work?"**

```text
RAG is a technique where we retrieve relevant information
from an external knowledge source and provide that information
to the LLM as context before generating the answer.

First, in the indexing pipeline, we load the documents,
extract the text, split it into smaller chunks, create embeddings
for those chunks, and store the embeddings along with the text
and metadata in a vector database.

At runtime, when the user asks a question, we create an embedding
for the query and use similarity search to retrieve the most
relevant chunks.

Then we combine the user query with those retrieved chunks to
create an augmented prompt and send it to the LLM.

The LLM then generates the final answer using the retrieved context.
```

---

# 🧠 **Interview: Why Do We Need Chunking?**

```text
We use chunking because sending an entire large document
to the LLM can exceed its context window.

Chunking also lets the application retrieve only the relevant
pieces instead of passing the complete document every time.
This can reduce unnecessary context and token usage.
```

---

# 🧠 **Interview: Why Do We Need Embeddings?**

```text
The user query and document chunks are initially text,
but similarity search works on numerical representations.

So we convert both the query and document chunks into vectors
using an embedding model.

Then we can compare the vectors using a similarity measure
such as cosine similarity and retrieve relevant chunks.
```

---

# 🧠 **Interview: Why Do We Need a Vector Database?**

```text
A vector database provides a place to store embeddings,
along with the original text and metadata, and search for
vectors that are similar to the query vector.
```

---

# 🧪 **Simple RAG Development Workflow**

```text
1. Identify Knowledge Source
        ↓
2. Load Documents
        ↓
3. Extract Text
        ↓
4. Split Documents
        ↓
5. Create Embeddings
        ↓
6. Store in Vector DB
        ↓
7. Create Retriever
        ↓
8. Embed User Query
        ↓
9. Retrieve Relevant Chunks
        ↓
10. Build Prompt
        ↓
11. Generate Answer
        ↓
12. Evaluate Retrieval + Answer
```

---

# 🧱 **RAG Component Responsibilities**

| Component | Responsibility |
|---|---|
| **Document Loader** | Load / extract source content |
| **Document** | Store content + metadata |
| **Chunking** | Split large content into smaller units |
| **Embedding Model** | Convert text into vectors |
| **Vector DB** | Store and search vectors + data |
| **Retriever** | Fetch relevant chunks |
| **Prompt / Augmentation** | Combine query + retrieved context |
| **LLM** | Generate the final answer |

**Memory trick:**

```text
Loader
→ Get the data

Chunking
→ Split the data

Embedding
→ Convert to vectors

Vector DB
→ Store + search

Retriever
→ Find relevant data

Augmentation
→ Add context to query

LLM
→ Generate answer
```

---

# 🆚 **RAG Architecture at a Glance**

| Stage | Input | Processing | Output |
|---|---|---|---|
| **Extraction** | Raw document | Loader | Document objects |
| **Chunking** | Large document | Split | Chunks |
| **Embedding** | Text chunks | Embedding model | Vectors |
| **Indexing** | Vectors + text | Vector DB | Searchable index |
| **Retrieval** | Query | Similarity search | Relevant chunks |
| **Augmentation** | Query + chunks | Prompt creation | Augmented prompt |
| **Generation** | Augmented prompt | LLM | Answer |

---

# 📋 **Quick Revision**

```text
RAG
→ Retrieval-Augmented Generation

Private Data
→ External / application data not automatically known by the model

In-Context Learning
→ Providing context together with the question

Document Loader
→ Loads source data into Document objects

PyPDFLoader
→ Loads PDF content

Document
→ Standard object containing content + metadata

page_content
→ Extracted text

metadata
→ Source / descriptive information

Context Window
→ Maximum token context available to a model request

Chunking
→ Splitting large documents into smaller pieces

Embedding Model
→ Converts text into vectors

Cosine Similarity
→ Measures directional similarity between vectors

Vector Database
→ Stores and searches embeddings / vectors

Indexing
→ Preparing documents for retrieval

Retriever
→ Retrieves relevant chunks

Top-k
→ Highest-scoring retrieved chunks

Augmentation
→ Combines retrieved context with the user query

Generation
→ Uses the LLM to generate the answer
```

---

# 🧠 **Memory Trick**

```text
INDEX
→ Prepare the knowledge

RETRIEVE
→ Find relevant knowledge

AUGMENT
→ Add knowledge to the query

GENERATE
→ Ask the LLM to answer
```

**The shortest formula is:**

```text
RAG
=
Retrieve
+
Augment
+
Generate
```

---

# 📈 **RAG Quality Mental Model**

A RAG application should not be judged only by its LLM.

**Consider:**

```text
Document Quality
+
Chunking Quality
+
Embedding Quality
+
Retrieval Quality
+
Prompt Quality
+
Generation Quality
```

**A useful conceptual formulation is:**

$$
\text{RAG Quality}
\approx
f(
\text{Retrieval},
\text{Context},
\text{Prompt},
\text{Generation}
)
$$

---

# 🏆 **Final RAG Architecture**

```text
                    ┌──────────────────────┐
                    │    KNOWLEDGE SOURCE  │
                    └──────────┬───────────┘
                               ↓
                         DOCUMENT LOADER
                               ↓
                          TEXT EXTRACTION
                               ↓
                            CHUNKING
                               ↓
                        EMBEDDING MODEL
                               ↓
                             VECTORS
                               ↓
                        ┌───────────────┐
                        │  VECTOR DB    │
                        │               │
                        │ Vector        │
                        │ Text          │
                        │ Metadata      │
                        └───────┬───────┘
                                │
                                │
                           USER QUERY
                                │
                                ↓
                        EMBEDDING MODEL
                                ↓
                           QUERY VECTOR
                                ↓
                        SIMILARITY SEARCH
                                ↓
                        RELEVANT CHUNKS
                                ↓
                         QUERY + CONTEXT
                                ↓
                          AUGMENTATION
                                ↓
                               LLM
                                ↓
                             RESPONSE
```

---

# 🏁 **Key Takeaways**

> **1. An LLM is not automatically aware of private or external documents available inside an application.**

> **2. In-context learning provides relevant information alongside the user's question.**

> **3. Sending a large document in one prompt can exceed the model's context window.**

> **4. Chunking splits large documents into smaller searchable pieces.**

> **5. Large context windows do not automatically remove the need for RAG because cost and context efficiency still matter.**

> **6. Context Rot describes the potential decline in effective model performance as context becomes larger.**

> **7. Embedding models convert document chunks and user queries into numerical vectors.**

> **8. Cosine similarity can be used to compare embeddings.**

> **9. A vector database stores embeddings together with useful text and metadata.**

> **10. Indexing prepares the knowledge base before users ask questions.**

> **11. Retrieval finds the most relevant chunks for the current query.**

> **12. Augmentation combines the user's query with retrieved context.**

> **13. Generation uses the LLM to create the final answer.**

> **14. RAG separates knowledge storage from model generation: the knowledge can live outside the model and be retrieved at runtime.**

---

# 📝 **One-Line Revision Formula**

```text
RAG
=
Document
→ Loader
→ Chunk
→ Embed
→ Store
→ Retrieve
→ Augment
→ Generate
```

**Or:**

```text
RAG
=
Retrieve
+
Augment
+
Generate
```

---

# 📚 **One-Page Revision**

```text
                         RAG
                          │
          ┌───────────────┴────────────────┐
          ↓                                ↓
      INDEXING                           QUERY
          ↓                                ↓
      Document                         User Query
          ↓                                ↓
       Loader                        Embedding
          ↓                                ↓
      Chunking                       Query Vector
          ↓                                ↓
     Embedding                      Similarity Search
          ↓                                ↓
    Vector Database                Relevant Chunks
                                           ↓
                                   Query + Context
                                           ↓
                                      Augmented Prompt
                                           ↓
                                          LLM
                                           ↓
                                        Answer
```

---

# 🎯 **Final Mental Model**

> **A RAG system separates knowledge preparation from question answering.**

```text
                    PREPARE KNOWLEDGE
                           ↓
                 Documents → Chunks
                           ↓
                      Embeddings
                           ↓
                      Vector DB
                           │
                           │
                           ↓
                     USER QUESTION
                           ↓
                     Query Embedding
                           ↓
                   Retrieve Relevant Data
                           ↓
                    Add Context to Query
                           ↓
                           LLM
                           ↓
                         ANSWER
```

> **The key idea is not to make the LLM memorize every piece of information. Instead, retrieve the right information at runtime and give it to the LLM as context.**

---

# 🔗 **Useful Resource**

> **LangChain APIs and package names can evolve. Check the current official documentation when implementing the architecture in a new project.**

```text
LangChain Documentation
https://docs.langchain.com
```

---

# 🏁 **End Note**

```text
Load the knowledge
      ↓
Split the knowledge
      ↓
Convert it into vectors
      ↓
Store it
      ↓
Find relevant pieces
      ↓
Add them to the query
      ↓
Generate with the LLM
```

> **RAG = Retrieve the right knowledge → Augment the prompt → Generate the answer.**
